# 🥣 Ingredient Identification & Comprehensive Nutrition Version 3 Training
### PyTorch Multi-Task Deep Neural Network (231 Food Categories — Colab Auto-Save & Resume Ready)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Incredient_Identification/NoteBooks/Ingredient%20Identification/Version3/ingredient_identification_v3_training.ipynb)

---
### 🚀 What's New in Version 3:
- **Aligned 1-to-1 with Food Detection V3**: Supports all **231 food classes (`0..230`)**, including **Poori, Chutney, Momos, Pani puri, Jalebi, Vada pav, Pav, Naan, Cutlet, and Poha**.
- **328 Unique Ingredients & 11 Nutrition Targets**: Multi-label ingredient classification + normalized macro/micronutrient regression across **924 samples** (`462 train / 231 val / 231 test`).
- **Disconnect-Proof Google Drive Auto-Save & Resume**: Atomically saves `last.pt` (plus `last_backup.pt`) and `best.pt` to Google Drive every epoch, with 1-click resume if Colab disconnects.

## Step 1: Install Dependencies, Verify Hardware & Mount Google Drive

In [ ]:
!pip install -q torch torchvision pandas numpy matplotlib pyyaml scikit-learn

import torch, sys
print('=' * 60)
print('🖥️  SYSTEM & HARDWARE SPECIFICATIONS (INGREDIENT V3)')
print('=' * 60)
print(f'Python Version   : {sys.version.split()[0]}')
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {"✅ Yes" if torch.cuda.is_available() else "❌ No (CPU Mode)"}')
if torch.cuda.is_available():
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
print('=' * 60)

if 'google.colab' in str(get_ipython()):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive mounted! Checkpoints will sync safely to Drive.')
    except Exception as e:
        print(f'Drive note: {e}')

## Step 2: Fresh GitHub Repository Clone / Pull

In [ ]:
import os

REPO_URL = 'https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git'
REPO_NAME = 'SouthIndian_Food_Nutrition'

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f'/content/{REPO_NAME}'):
        print(f'🚀 Cloning {REPO_NAME} from GitHub...')
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        print(f'🔄 Updating {REPO_NAME}...')
        !git -C /content/{REPO_NAME} pull --ff-only
    %cd /content/{REPO_NAME}
    print(f'✅ Active Working Directory: {os.getcwd()}')
else:
    print(f'Running in local workspace: {os.getcwd()}')

## Step 3: Load Version 3 Dataset (231 Food Categories)

In [ ]:
import os, pandas as pd

candidate_dirs = [
    'Incredient_Identification/DataSet/Version3 Used Data',
    os.path.join(os.getcwd(), 'Incredient_Identification', 'DataSet', 'Version3 Used Data'),
    '/content/SouthIndian_Food_Nutrition/Incredient_Identification/DataSet/Version3 Used Data'
]

DATASET_DIR = next((os.path.abspath(d) for d in candidate_dirs if os.path.exists(d)), None)
if not DATASET_DIR:
    raise FileNotFoundError('❌ Version 3 ingredient dataset directory not found!')

print(f'📁 Dataset Directory: {DATASET_DIR}')
train_df = pd.read_csv(os.path.join(DATASET_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATASET_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(DATASET_DIR, 'test.csv'))
vocab_df = pd.read_csv(os.path.join(DATASET_DIR, 'ingredients_vocabulary_v3.csv'))
classes_df = pd.read_csv(os.path.join(DATASET_DIR, 'food_and_nutrition_master_v3.csv'))

NUM_FOODS = len(classes_df)
NUM_INGREDIENTS = len(vocab_df)

print('=' * 60)
print(f'🔥 Total Food Classes      : {NUM_FOODS}')
print(f'🌿 Total Unique Ingredients: {NUM_INGREDIENTS}')
print(f'📊 Train Samples           : {len(train_df)}')
print(f'📊 Val Samples             : {len(val_df)}')
print(f'📊 Test Samples            : {len(test_df)}')
print('=' * 60)

## Step 4: Dataset Class & PyTorch DataLoader

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

ing2id = {r['ingredient_key'].lower(): int(r['ingredient_id']) for _, r in vocab_df.iterrows()}
id2ing = {int(r['ingredient_id']): r['display_name'] for _, r in vocab_df.iterrows()}
food2id = {r['food_name']: int(r['food_id']) for _, r in classes_df.iterrows()}
id2food = {int(r['food_id']): r['food_name'] for _, r in classes_df.iterrows()}

NUT_COLS = [
    'calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g', 'fiber_g',
    'sugar_g', 'sodium_mg', 'potassium_mg', 'calcium_mg', 'iron_mg', 'vitamin_c_mg'
]

class RecipeDataset(Dataset):
    def __init__(self, df, nut_mean=None, nut_std=None):
        self.food_ids = torch.tensor(df['food_id'].values, dtype=torch.long)
        labels = np.zeros((len(df), NUM_INGREDIENTS), dtype=np.float32)
        for idx, tokens in enumerate(df['ingredients_tokens']):
            for t in str(tokens).split(';'):
                t_clean = t.strip().lower()
                if t_clean in ing2id:
                    labels[idx, ing2id[t_clean]] = 1.0
        self.ing_targets = torch.tensor(labels, dtype=torch.float32)

        nut_vals = df[NUT_COLS].values.astype(np.float32)
        if nut_mean is None:
            self.nut_mean = torch.tensor(nut_vals.mean(axis=0), dtype=torch.float32)
            self.nut_std = torch.tensor(nut_vals.std(axis=0) + 1e-6, dtype=torch.float32)
        else:
            self.nut_mean = nut_mean
            self.nut_std = nut_std
        self.nut_targets = (torch.tensor(nut_vals, dtype=torch.float32) - self.nut_mean) / self.nut_std

    def __len__(self):
        return len(self.food_ids)

    def __getitem__(self, idx):
        return self.food_ids[idx], self.ing_targets[idx], self.nut_targets[idx]

train_ds = RecipeDataset(train_df)
val_ds = RecipeDataset(val_df, train_ds.nut_mean, train_ds.nut_std)
test_ds = RecipeDataset(test_df, train_ds.nut_mean, train_ds.nut_std)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'✅ DataLoaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches.')

## Step 5: Define Multi-Task Neural Network Architecture

In [ ]:
import torch.nn as nn

class SouthIndianIngredientNutritionModel(nn.Module):
    def __init__(self, num_foods=231, num_ingredients=328, embed_dim=64, num_nutrition=11):
        super().__init__()
        self.food_embedding = nn.Embedding(num_foods, embed_dim)
        self.shared_trunk = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.ingredient_head = nn.Sequential(
            nn.Linear(128, 96),
            nn.ReLU(),
            nn.Linear(96, num_ingredients)
        )
        self.nutrition_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_nutrition)
        )

    def forward(self, food_ids):
        embed = self.food_embedding(food_ids)
        shared = self.shared_trunk(embed)
        return self.ingredient_head(shared), self.nutrition_head(shared)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SouthIndianIngredientNutritionModel(NUM_FOODS, NUM_INGREDIENTS, embed_dim=64, num_nutrition=len(NUT_COLS)).to(device)
print(model)

## Step 6: Disconnect-Safe Training & Auto-Resume (Google Drive Sync)

In [ ]:
import os, shutil, torch, torch.optim as optim

EPOCHS = 100
LR = 0.002
AUTO_RESUME = True

LOCAL_OUT_DIR = 'Incredient_Identification/Models/Ingredient Identification/Version3'
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)
BEST_WEIGHTS_PATH = os.path.join(LOCAL_OUT_DIR, 'best.pt')
LAST_WEIGHTS_PATH = os.path.join(LOCAL_OUT_DIR, 'last.pt')

HAS_DRIVE = os.path.exists('/content/drive/MyDrive')
DRIVE_OUT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Ingredient_Identification'
if HAS_DRIVE:
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

def atomic_save(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_copy(src, dst):
    if not os.path.exists(src):
        return
    tmp = dst + '.tmp'
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

bce_loss = nn.BCEWithLogitsLoss()
mse_loss = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

start_epoch = 1
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

if AUTO_RESUME:
    ckpt_candidates = [
        os.path.join(DRIVE_OUT_DIR, 'last.pt'),
        os.path.join(DRIVE_OUT_DIR, 'last_backup.pt'),
        LAST_WEIGHTS_PATH,
    ]
    for p in ckpt_candidates:
        if os.path.exists(p) and os.path.getsize(p) > 1024:
            try:
                ckpt = torch.load(p, map_location=device, weights_only=False)
                model.load_state_dict(ckpt['model_state_dict'])
                if 'optimizer_state_dict' in ckpt:
                    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                start_epoch = int(ckpt.get('epoch', 0)) + 1
                best_val_loss = float(ckpt.get('best_val_loss', ckpt.get('val_loss', float('inf'))))
                history = ckpt.get('history', history)
                print(f'🔄 Resumed from {p} at Epoch {start_epoch}/{EPOCHS} (best_val={best_val_loss:.4f})')
                break
            except Exception as e:
                print(f'⚠️ Skipping {p}: {e}')
    d_best = os.path.join(DRIVE_OUT_DIR, 'best.pt')
    if HAS_DRIVE and os.path.exists(d_best) and not os.path.exists(BEST_WEIGHTS_PATH):
        atomic_copy(d_best, BEST_WEIGHTS_PATH)

print('=' * 65)
print(f'🚀 TRAINING INGREDIENT & NUTRITION PREDICTOR V3 ({start_epoch}..{EPOCHS})')
print('=' * 65)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for f_ids, ing_tg, nut_tg in train_loader:
        f_ids, ing_tg, nut_tg = f_ids.to(device), ing_tg.to(device), nut_tg.to(device)
        optimizer.zero_grad()
        pred_ing, pred_nut = model(f_ids)
        loss = bce_loss(pred_ing, ing_tg) * 2.0 + mse_loss(pred_nut, nut_tg)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(f_ids)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for f_ids, ing_tg, nut_tg in val_loader:
            f_ids, ing_tg, nut_tg = f_ids.to(device), ing_tg.to(device), nut_tg.to(device)
            pred_ing, pred_nut = model(f_ids)
            loss = bce_loss(pred_ing, ing_tg) * 2.0 + mse_loss(pred_nut, nut_tg)
            val_loss += loss.item() * len(f_ids)
    val_loss /= len(val_ds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    payload = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'food2idx': food2id,
        'ing2idx': ing2id,
        'nut_mean': train_ds.nut_mean.tolist(),
        'nut_std': train_ds.nut_std.tolist(),
        'num_foods': NUM_FOODS,
        'num_ingredients': NUM_INGREDIENTS,
        'nut_cols': NUT_COLS,
        'epoch': epoch,
        'val_loss': val_loss,
        'best_val_loss': min(best_val_loss, val_loss),
        'history': history,
    }

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        payload['best_val_loss'] = best_val_loss
        atomic_save(payload, BEST_WEIGHTS_PATH)
        if HAS_DRIVE:
            atomic_copy(BEST_WEIGHTS_PATH, os.path.join(DRIVE_OUT_DIR, 'best.pt'))

    atomic_save(payload, LAST_WEIGHTS_PATH)
    if HAS_DRIVE:
        d_last = os.path.join(DRIVE_OUT_DIR, 'last.pt')
        if os.path.exists(d_last):
            atomic_copy(d_last, os.path.join(DRIVE_OUT_DIR, 'last_backup.pt'))
        atomic_copy(LAST_WEIGHTS_PATH, d_last)

    if epoch % 10 == 0 or epoch == EPOCHS or epoch == start_epoch:
        print(f'Epoch [{epoch:3d}/{EPOCHS}] | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Best Val: {best_val_loss:.4f}')

print('\n✅ Version 3 Training Finished!')
print(f'🏆 Best Weights: {BEST_WEIGHTS_PATH}')

## Step 7: Plot Training & Validation Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.plot(history['train_loss'], label='Train Combined Loss', color='#2ecc71', lw=2.5)
ax1.plot(history['val_loss'], label='Val Combined Loss', color='#e74c3c', lw=2.5, linestyle='--')
ax1.set_title('Dual-Head Training & Validation Loss (V3)', fontsize=14)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

epochs_range = range(1, len(history['val_loss']) + 1)
ax2.plot(epochs_range, history['val_loss'], label='Validation Convergence', color='#3498db', lw=2.5)
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
best_loss = min(history['val_loss'])
ax2.scatter([best_epoch], [best_loss], color='#e67e22', s=120, zorder=5, label=f'Best Val: {best_loss:.4f} (Ep {best_epoch})')
ax2.set_title('Validation Loss Optimization (V3)', fontsize=14)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Validation Loss')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.savefig('ingredient_v3_loss_curve.png', dpi=200)
plt.show()

## Step 8: Test Inference on Existing & New Version 3 Food Classes

In [ ]:
ckpt = torch.load(BEST_WEIGHTS_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

sample_dishes = ['Masala dosa', 'Chicken biryani', 'Poori', 'Momos', 'Pani puri', 'Vada pav', 'Poha']
nut_mean_t = torch.tensor(ckpt['nut_mean'], device=device)
nut_std_t = torch.tensor(ckpt['nut_std'], device=device)

print('=' * 65)
print('🍽️  INFERENCE VERIFICATION ON V3 TEST DISHES')
print('=' * 65)

for dish in sample_dishes:
    if dish in food2id:
        fid = torch.tensor([food2id[dish]], device=device)
        with torch.no_grad():
            pred_ing, pred_nut = model(fid)
            probs = torch.sigmoid(pred_ing[0]).cpu().numpy()
            unnorm_nut = (pred_nut[0] * nut_std_t + nut_mean_t).cpu().numpy()
        top_indices = np.where(probs >= 0.25)[0]
        if len(top_indices) == 0:
            top_indices = np.argsort(probs)[-4:]
        pred_ingredients = [id2ing.get(int(idx), str(idx)) for idx in top_indices]
        print(f'\nFood: {dish}')
        print(f'  Predicted Ingredients: {", ".join(pred_ingredients)}')
        print(f'  Estimated Nutrition  : Calories: {unnorm_nut[0]:.1f} kcal, Protein: {unnorm_nut[1]:.1f}g, Carbs: {unnorm_nut[2]:.1f}g, Fat: {unnorm_nut[3]:.1f}g')

## Step 9: Save `ingredient_identification_v3_best.pt` to Drive & 1-Click Download

In [ ]:
import os, shutil

STANDARDIZED_NAME = 'ingredient_identification_v3_best.pt'
shutil.copy2(BEST_WEIGHTS_PATH, STANDARDIZED_NAME)
shutil.copy2(BEST_WEIGHTS_PATH, os.path.join(LOCAL_OUT_DIR, STANDARDIZED_NAME))

if HAS_DRIVE:
    for f in (STANDARDIZED_NAME, 'ingredient_v3_loss_curve.png'):
        if os.path.exists(f):
            atomic_copy(f, os.path.join(DRIVE_OUT_DIR, f))
            print(f'✅ Saved to Google Drive: {DRIVE_OUT_DIR}/{f}')

try:
    from google.colab import files
    files.download(STANDARDIZED_NAME)
except ImportError:
    print(f'Running locally: Model saved as {STANDARDIZED_NAME}')